# 02 — Understat Data Collection

Collects xG/xA data from Understat and saves raw files.
Then fuzzy-matches Understat player names to FPL player names
and builds a unified dataset.

**Outputs:**
- `data/raw/understat_players.parquet` — season totals per player
- `data/raw/understat_matches.parquet` — match-by-match xG/xA
- `data/processed/player_id_map.parquet` — FPL ↔ Understat ID mapping
- `data/processed/merged_players.parquet` — unified dataset for modelling

## 0. Smoke test — verify Understat endpoints are reachable

In [ ]:
import asyncio
import sys
sys.path.insert(0, '..')

from src.data.understat_client import UnderstatClient

async def smoke_test():
    async with UnderstatClient() as client:
        players = await client.get_league_players(season='2025')
        print(f'Season totals: {len(players)} players')
        print(f'Columns: {list(players.columns)}')
        print(players.head(3).to_string(index=False))

await smoke_test()

## 1. Imports & setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

## 2. Fetch season totals

In [ ]:
async def fetch_season_totals():
    async with UnderstatClient() as client:
        return await client.get_league_players(season='2025')

understat_players = await fetch_season_totals()
print(understat_players.shape)
understat_players.head()

## 3. Fetch match-by-match data for all players

Uses 10 concurrent workers with 100ms delay. ~600 players → ~2-3 minutes.

In [ ]:
async def fetch_all_matches(player_ids):
    async with UnderstatClient(concurrency=10, request_delay=0.1) as client:
        return await client.get_all_player_matches(
            player_ids,
            season_filter='2025',
        )

player_ids = understat_players['id'].tolist()
understat_matches = await fetch_all_matches(player_ids)

print(f'Match rows: {len(understat_matches)}')
print(f'Unique players: {understat_matches["understat_id"].nunique()}')
understat_matches.head()

In [ ]:
# Basic sanity checks
print('Date range:', understat_matches['date'].min(), '→', understat_matches['date'].max())
print('Seasons present:', understat_matches['season'].unique())
print('Null xG:', understat_matches['xG'].isna().sum())
print('Null xA:', understat_matches['xA'].isna().sum())

## 4. Save raw Understat files

In [ ]:
understat_players.to_parquet(RAW / 'understat_players.parquet', index=False)
understat_matches.to_parquet(RAW / 'understat_matches.parquet', index=False)

print('Saved:')
print(f'  understat_players.parquet  — {len(understat_players)} rows')
print(f'  understat_matches.parquet  — {len(understat_matches)} rows')

## 5. Fuzzy name matching — FPL ↔ Understat

The core challenge: FPL uses short names (`"Salah"`, `"Trent"`) while Understat
uses full names (`"Mohamed Salah"`, `"Trent Alexander-Arnold"`). We can't join on
exact string match — we need fuzzy matching.

**Strategy:**
- Build a full name for each FPL player from `first_name + second_name`
- Use `rapidfuzz.process.extractOne` with `token_sort_ratio` (handles word-order differences)
- Score ≥ 85 → auto-match
- Score 70–84 → flag for manual review
- Manual override dict for known hard cases

In [ ]:
from rapidfuzz import process, fuzz

# Load FPL players
fpl_players = pd.read_parquet(RAW / 'fpl_players.parquet')
fpl_players['full_name'] = fpl_players['first_name'] + ' ' + fpl_players['second_name']

print('FPL players:', len(fpl_players))
print('Understat players:', len(understat_players))

In [ ]:
# Manual overrides for known mismatches
# Format: {understat_name: fpl_full_name}
MANUAL_OVERRIDES = {
    'Son Heung-min': 'Heung-Min Son',
    'Matheus Nunes': 'Matheus Nunes',
    'Gabriel Magalhaes': 'Gabriel Magalhães',
    'Pedro Porro': 'Pedro Lomba Porro',
    'Emile Smith Rowe': 'Emile Smith Rowe',
}

fpl_names = fpl_players['full_name'].tolist()
fpl_name_to_id = dict(zip(fpl_players['full_name'], fpl_players['id']))

records = []

for _, row in understat_players.iterrows():
    u_name = row['player_name']
    u_id = row['id']

    # Apply manual override if present
    if u_name in MANUAL_OVERRIDES:
        fpl_name = MANUAL_OVERRIDES[u_name]
        fpl_id = fpl_name_to_id.get(fpl_name)
        score = 100
    else:
        match = process.extractOne(
            u_name,
            fpl_names,
            scorer=fuzz.token_sort_ratio,
        )
        if match:
            fpl_name, score, _ = match
            fpl_id = fpl_name_to_id.get(fpl_name)
        else:
            fpl_name, score, fpl_id = None, 0, None

    records.append({
        'understat_id': u_id,
        'understat_name': u_name,
        'fpl_id': fpl_id,
        'fpl_name': fpl_name,
        'match_score': score,
        'match_status': 'auto' if score >= 85 else ('review' if score >= 70 else 'no_match'),
    })

id_map = pd.DataFrame(records)
print('Total:', len(id_map))
print(id_map['match_status'].value_counts())

In [ ]:
# Inspect players flagged for review
review = id_map[id_map['match_status'] == 'review']
print(f'{len(review)} players need review:')
review[['understat_name', 'fpl_name', 'match_score']].sort_values('match_score', ascending=False)

In [ ]:
# Inspect no-matches (likely non-EPL players in Understat who aren't in FPL)
no_match = id_map[id_map['match_status'] == 'no_match']
print(f'{len(no_match)} players with no FPL match (expected — Understat includes players not in FPL):')
no_match[['understat_name', 'fpl_name', 'match_score']].head(20)

In [ ]:
# Keep only confirmed matches for the merge
id_map_confirmed = id_map[id_map['match_status'] == 'auto'].copy()
print('Confirmed matches:', len(id_map_confirmed))

id_map.to_parquet(PROCESSED / 'player_id_map.parquet', index=False)
print('Saved player_id_map.parquet')

## 6. Assign gameweek numbers to Understat matches

FPL data is indexed by `round` (gameweek 1–38). Understat data has match `date`.
We assign each Understat match to an FPL gameweek by finding which gameweek
deadline the match date falls between.

In [ ]:
from src.data.fpl_client import FPLClient

fpl = FPLClient()
events = fpl.get_events()
events['deadline_time'] = pd.to_datetime(events['deadline_time'], utc=True)

# Build deadline lookup: gw → deadline
gw_deadlines = events[['id', 'deadline_time']].sort_values('id').reset_index(drop=True)
gw_deadlines.columns = ['round', 'deadline']
gw_deadlines.head()

In [ ]:
def assign_gameweek(match_date, deadlines_df):
    """
    Return the FPL gameweek number for a given match date.
    A match belongs to gameweek N if it falls after the GW N deadline
    and before the GW N+1 deadline.
    """
    if pd.isna(match_date):
        return None
    match_date = match_date.tz_localize('UTC') if match_date.tzinfo is None else match_date
    mask = deadlines_df['deadline'] <= match_date
    eligible = deadlines_df[mask]
    if eligible.empty:
        return None
    return int(eligible.iloc[-1]['round'])

understat_matches['round'] = understat_matches['date'].apply(
    lambda d: assign_gameweek(d, gw_deadlines)
)

print('Gameweek assignment coverage:')
print(f'  Assigned: {understat_matches["round"].notna().sum()}')
print(f'  Unassigned: {understat_matches["round"].isna().sum()}')
understat_matches[['date', 'round', 'h_team', 'a_team', 'xG', 'xA']].head()

## 7. Merge FPL gameweeks + Understat matches

In [ ]:
# Load FPL gameweek data
fpl_gw = pd.read_parquet(RAW / 'fpl_gameweeks.parquet')
print('FPL gameweek rows:', len(fpl_gw))

# Add understat_id to FPL data via id_map
fpl_gw = fpl_gw.merge(
    id_map_confirmed[['fpl_id', 'understat_id']],
    left_on='player_id',
    right_on='fpl_id',
    how='left',
).drop(columns=['fpl_id'])

print(f'Players with understat_id: {fpl_gw["understat_id"].notna().sum()} / {len(fpl_gw)}')

In [ ]:
# Aggregate Understat match data to one row per player per gameweek
# (some players have double gameweeks — sum xG/xA, take max minutes)
understat_agg = (
    understat_matches
    .dropna(subset=['round'])
    .groupby(['understat_id', 'round'])
    .agg(
        xG=('xG', 'sum'),
        xA=('xA', 'sum'),
        shots=('shots', 'sum'),
        key_passes=('key_passes', 'sum'),
        npxG=('npxG', 'sum'),
        xGChain=('xGChain', 'sum'),
        xGBuildup=('xGBuildup', 'sum'),
        us_minutes=('time', 'sum'),
    )
    .reset_index()
)

understat_agg['round'] = understat_agg['round'].astype(int)
print('Understat aggregated rows:', len(understat_agg))

In [ ]:
# Merge on (understat_id, round)
merged = fpl_gw.merge(
    understat_agg,
    on=['understat_id', 'round'],
    how='left',
)

print('Merged shape:', merged.shape)
print('xG coverage:', f"{merged['xG'].notna().mean():.1%} of rows have Understat data")
merged.head()

In [ ]:
# Add player metadata (position, team, name, price)
merged = merged.merge(
    fpl_players[['id', 'web_name', 'position', 'team', 'now_cost']],
    left_on='player_id',
    right_on='id',
    how='left',
).drop(columns=['id'])

print('Final shape:', merged.shape)
print('Columns:', list(merged.columns))

## 8. Save merged dataset

In [ ]:
merged.to_parquet(PROCESSED / 'merged_players.parquet', index=False)

print('Saved merged_players.parquet')
print(f'  Rows: {len(merged)}')
print(f'  Columns: {len(merged.columns)}')
print(f'  Players: {merged["player_id"].nunique()}')
print(f'  Gameweeks: {merged["round"].nunique()}')
print(f'  xG coverage: {merged["xG"].notna().mean():.1%}')